# 声明文件与第三方类型

学习目标：能为已有 JavaScript 实现写准确声明，区分模块与全局环境声明，并从独立消费者检查类型和运行行为。

前置知识：函数与类、模块导入导出、ESM、泛型、类型和值、Node.js 与 npm 项目依赖。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict；另启用 allowJs。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/18-declaration-files/。

1. [meter/index.js](scripts/18-declaration-files/meter/index.js)：配套实现与示例。
2. [meter/index.d.ts](scripts/18-declaration-files/meter/index.d.ts)：类型声明。
3. [main.ts](scripts/18-declaration-files/main.ts)：配套实现与示例。
4. [host.d.ts](scripts/18-declaration-files/host.d.ts)：类型声明。
5. [host-init.js](scripts/18-declaration-files/host-init.js)：配套实现与示例。
6. [ambient.d.ts](scripts/18-declaration-files/ambient.d.ts)：类型声明。
7. [missing-runtime.ts](scripts/18-declaration-files/missing-runtime.ts)：配套实现与示例。
8. [meter/package.json](scripts/18-declaration-files/meter/package.json)：配套实现与示例。
9. [consumer.ts](scripts/18-declaration-files/consumer.ts)：配套实现与示例。
10. [pack-check.mjs](scripts/18-declaration-files/pack-check.mjs)：配套实现与示例。
11. [tsconfig.json](scripts/18-declaration-files/tsconfig.json)：本章独立项目配置。
12. [type-errors.ts](scripts/18-declaration-files/type-errors.ts)、[tsconfig.errors.json](scripts/18-declaration-files/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:18
```

Step 2：生成本章 JavaScript。

```bash
npm run build:18
```

Step 3：运行本章正常示例。

```bash
npm run run:18
# 正常退出；各段预期输出见代码注释。
```

同一文件的片段按正文顺序衔接，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 声明描述存在，不提供实现

.d.ts 文件只参与类型检查，不生成 JavaScript。环境声明（ambient declaration）使用 declare 描述由其他代码或宿主提供的实体；declare function 没有函数体，declare class 的方法也只写签名。

先看自己的 JavaScript 模块：label 接受数值并返回字符串，Meter 接受初值并累计步长。声明必须与这些实际导出、参数和返回值一致。

对应 [meter/index.js](scripts/18-declaration-files/meter/index.js)。

```javascript
export function label(value) { return `读数:${value}`; }
export class Meter {
  constructor(start) { this.value = start; }
  add(step) { this.value += step; return this.value; }
}
```

## 2 为函数和类写模块声明文件

与 index.js 相邻的 index.d.ts 声明其导出；顶层 export 使这些名字属于模块，不进入全局。类的构造参数、实例属性、方法返回类型各自写清楚，不用 any 或任意属性索引来消除不明确的错误。

模块实现的存在与类型声明的正确是两个条件。下面声明为消费者提供签名，本身不会创建 Meter 构造函数。

对应 [meter/index.d.ts](scripts/18-declaration-files/meter/index.d.ts)。

```typescript
export declare function label(value: number): string;
export declare class Meter {
  constructor(start: number);
  value: number;
  add(step: number): number;
}
```

## 3 消费相邻声明与宿主类型

“能找到类型”与“能加载实现”应分别检查。下面用同一个 meter 模块区分两个阶段。

本例同一个 ./meter/index.js 导入有两个观察阶段：TypeScript 用相邻 .d.ts 检查调用，Node 用 .js 中的真实实现执行。allowJs 把已有实现复制到输出目录，checkJs 为 false，因此不能把消费者通过类型检查误称为已经核验了 JavaScript 函数体。

Map 等标准语言 API 的类型来自内置 lib；node:path 这样的宿主模块由 @types/node 描述。本章 lib 为 ES2025，types 显式列 node；这些配置都不会给宿主补实现。

![声明与实现分别服务检查器和运行时。导入说明符相同，但两个阶段读取的文件不同。](image/illustration/18-01-declaration-runtime-files.svg)

图示说明：图限定为本例的相邻声明解析，不展开第三方包的全部搜索规则。

阅读 main.ts 的导入时，先找到前面声明的签名，再找到 index.js 的函数体；运行输出用于核对实现，不能由声明推算为已运行。

对应 [main.ts](scripts/18-declaration-files/main.ts)。

```typescript
import { Meter, label } from "./meter/index.js";
import { basename } from "node:path";
import "./host-init.js";
const meter = new Meter(5);
console.log(label(meter.add(2)), basename("reading.txt"));
console.log(chapter18Title);
globalThis.chapter18Title = undefined; // 清除本进程设置的宿主示例值。
// 预期输出：读数:7 reading.txt
// 预期输出：宿主值
```

## 4 全局声明与真实初始化

没有顶层导入导出的 .d.ts 可声明全局实体。这里 host.d.ts 只在本章 files 中列入；其他章节不会因文件存在就自动得到这个全局名。

允许 undefined 是因为使用前未必完成初始化。声明本身不赋值，消费者先加载 host-init.js，运行完成后撤销值；真实宿主 API 的可用时机也需另行确认。

对应 [host.d.ts](scripts/18-declaration-files/host.d.ts)。

```typescript
declare var chapter18Title: string | undefined;
```

## 5 初始化由宿主或实现完成

这个短实现模拟宿主注入值，与上一节的全局声明分开。必须在使用前实际执行它，不能用引用 .d.ts 的方式代替初始化。

对应 [host-init.js](scripts/18-declaration-files/host-init.js)。

```javascript
globalThis.chapter18Title = "宿主值";
```

## 6 环境模块声明与宽泛兜底的风险

declare module 加字符串名称可描述宿主提供但没有相应源文件的模块。它应位于脚本形式的声明文件中；如果同一文件顶层加了 import 或 export，通常会变成模块扩充，下一章会区分。

下面是一个“类型知道但当前宿主未实现”的教学反例。名称被准确声明为函数，而不是无条件接受所有内容；声明仍不能让 Node 找到不存在的模块。

对应 [ambient.d.ts](scripts/18-declaration-files/ambient.d.ts)。

```typescript
declare module "ts-b-missing-host" {
  export function readCount(): number;
}
```

## 7 类型通过而加载失败的反例

missing-runtime.ts 被纳入正常编译以观察声明的效力，但不进入正常运行入口。单独运行时，当前宿主没有提供这个包，Node 会在加载阶段拒绝。

Step 1：在完成 build:18 后运行缺失实现的反例。

```bash
node .build/18-declaration-files/missing-runtime.js
# 预期退出码 1，包含 ERR_MODULE_NOT_FOUND 和 ts-b-missing-host。
```

对应 [missing-runtime.ts](scripts/18-declaration-files/missing-runtime.ts)。

```typescript
import { readCount } from "ts-b-missing-host";
console.log(readCount()); // 声明允许调用，但不会安装或创建宿主模块。
```

## 8 包自带类型、@types 与查找入口

包可以自带声明，并用 package.json 的 types 或条件 exports 指向它；这时通常无需额外安装对应的 @types 包。未自带类型的 JavaScript 包可以由独立 @types 包描述，类型包版本仍要与实际实现核对。

本地 meter 包只含自己的两个文件，没有外部依赖。types 指向声明，运行入口指向实现，files 控制打包清单。它是本地教学包，private 防止误发布，不改变课程根依赖。

对应 [meter/package.json](scripts/18-declaration-files/meter/package.json)。

```json
{
  "name": "ts-b-meter",
  "version": "1.0.0",
  "private": true,
  "type": "module",
  "main": "./index.js",
  "types": "./index.d.ts",
  "files": ["index.js", "index.d.ts"]
}
```

## 9 从安装产物检查消费者

独立消费者只按包名导入，不能直接引用工作区实现。pack-check.mjs 将本地包打成 tarball，在临时消费者中离线安装，用项目内编译器检查，再运行生成代码；它同时核对类型解析确实命中安装目录中的 index.d.ts。

脚本在 scripts/18-declaration-files/ 下创建 .tmp-ts-b-declarations- 前缀的专属临时目录，把 npm 缓存也放入其中；完成或失败都会清理。consumer.ts 给出实际消费逻辑，打包辅助仅负责隔离和检查。安装、类型检查和运行是三个独立步骤。

Step 1：执行本地安装产物的消费者检查。

```bash
node scripts/18-declaration-files/pack-check.mjs
# 预期输出 installed-consumer 读数:12；随后输出 installed-types true。
```

对应 [consumer.ts](scripts/18-declaration-files/consumer.ts)。

```typescript
import { Meter, label } from "ts-b-meter";
const meter = new Meter(10);
console.log("installed-consumer", label(meter.add(2))); // → installed-consumer 读数:12
```

## 10 检查类型边界

下面的 [type-errors.ts](scripts/18-declaration-files/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
import { Meter, label } from "./meter/index.js";
label("12"); // 声明要求 number，不会把字符串自动转换。
new Meter("0"); // 构造参数与实现约定不符。
new Meter(0).add("2"); // 方法参数必须符合声明。
new Meter(0).reset(); // 不存在的能力不应靠宽泛声明放行。
// 预期诊断包含：TS2345, TS2339。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:18
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

声明是检查用的契约，不能创建实现。按实际模块形状写函数、类和环境声明，区分 lib、包自带类型与 @types；从安装后的消费者再次检查类型入口和运行入口。

## 练习

1. 给 Meter 的 JavaScript 实现新增 reset() 并同步声明；检查消费者可调用且运行后读数归零。

2. 临时把 label 声明的返回类型写错为 number，在消费者调用 toFixed；观察类型契约错误可能把问题推迟到运行时，练习后恢复声明。

3. 删除 ambient.d.ts 后重新检查 missing-runtime.ts，区分“编译器不认识模块”与“Node 找不到实现”的失败阶段。

## 参考与引用来源

- TypeScript 官方文档：[Type Declarations：.d.ts、lib 与外部类型](https://www.typescriptlang.org/docs/handbook/2/type-declarations.html)；[Declaration Reference：Functions、Classes、Globals](https://www.typescriptlang.org/docs/handbook/declaration-files/by-example.html)；[Modules .d.ts](https://www.typescriptlang.org/docs/handbook/declaration-files/templates/module-d-ts.html)；[Modules Reference：Ambient modules 与扩展名替换](https://www.typescriptlang.org/docs/handbook/modules/reference.html)；[Publishing：包含声明与类型依赖](https://www.typescriptlang.org/docs/handbook/declaration-files/publishing.html)；[types](https://www.typescriptlang.org/tsconfig/types.html)。
- npm：[npm pack](https://docs.npmjs.com/cli/v11/commands/npm-pack/)；[npm install：本地 tarball](https://docs.npmjs.com/cli/v11/commands/npm-install/)。
- Node.js 24.11.0：[fsPromises.rm](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisesrmpath-options)：临时目录清理；[Packages：包入口](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html)；[Child process：spawnSync](https://nodejs.org/download/release/v24.11.0/docs/api/child_process.html#child_processspawnsynccommand-args-options) 与 [maxBuffer 的字节上限](https://nodejs.org/download/release/v24.11.0/docs/api/child_process.html#maxbuffer-and-unicode)；[File system：临时目录与清理](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html)。